# Efficient Inference and Deployment


Lecture 12 was about the one big job of training a model. This lecture is about
the millions of small jobs that follow: answering requests from a trained model,
quickly and cheaply, on hardware that is rarely the hardware it was trained on.
The same roofline reasoning applies, but the answer flips: training is
compute-bound and inference, most of the time, is memory-bandwidth-bound. Almost
every technique in this lecture (KV caching, batching, paged memory,
quantization, speculative decoding, distillation) is a way of moving fewer bytes
per generated token. UDL does not cover this material; the notation follows the
primary papers, and each is cited where it is used. The two numbers to carry
out are the KV-cache size per token, $2\,L\,n_{kv}\,d_\text{head}$ values, and
the decode arithmetic intensity, about $B$ FLOP per byte for a batch of $B$
sequences.


## Today's roadmap

1. Why inference is its own problem: cost per token, latency metrics
2. Prefill vs. decode, arithmetic intensity, and the KV cache
3. Batching and serving systems: continuous batching, PagedAttention, prefix caching
4. Quantization: weights, activations, KV cache, formats and hardware; QLoRA
5. Speculative decoding
6. Smaller models: distillation, MoE serving, fine-tune vs. prompt vs. RAG
7. Practicalities: running an open-weight model, measuring tokens/s, cost
8. Summary


# Why inference is its own problem

Lecture 12 ended with the observation that a model is trained once and served
billions of times, which is why small models are overtrained. This section
makes that observation quantitative and introduces the vocabulary that serving
engineers use to describe "fast".


## Training is one job; serving is millions of them

- Training: one process, one objective, weeks long, every GPU busy by
  construction, batches of millions of tokens. The $6ND$ rule says what it
  costs (lecture 12).
- Serving: an unbounded stream of independent requests, each a few hundred to
  a few hundred thousand tokens, arriving at random, each wanting an answer
  *now*. The GPU is busy only if the serving system makes it busy.
- A model that is used at all is served for far more tokens than it was trained
  on. Llama 3 8B was trained on 15.6T tokens; a popular API serves that many
  tokens in days.
- So the unit of cost is the **generated token**, and the question of this
  lecture is: how many tokens per second does one GPU produce, at what
  latency, and at what loss of quality?


## Latency metrics

| Metric | Definition | What it measures |
|---|---|---|
| **TTFT** | time to first token | the prompt-processing (prefill) phase, plus queueing |
| **TPOT** | time per output token, after the first | the decode loop; its inverse is tokens/s *for one user* |
| **Latency** | $\text{TTFT} + n_\text{out} \times \text{TPOT}$ | the end-to-end time for one request |
| **Throughput** | tokens/s summed over all users on the GPU | the cost side: tokens per GPU-hour |
| **Goodput** | requests/s that *meet* a TTFT and TPOT target | the only number a service-level objective cares about |

Throughput and latency trade off through batching: a larger batch generates
more tokens per second in total and fewer per user. Serving is the art of
sitting at the right point of that trade-off. The goodput framing is from
[Zhong et al. 2024](https://arxiv.org/abs/2401.09670).


**The user-facing targets.** A person reads 5 to 7 tokens/s, so a TPOT of 50
to 100 ms streams faster than reading, and notices a pause of a few hundred
milliseconds, so TTFT for chat is under about a second. An **agent** (lecture
21) consumes every token over tens to hundreds of calls, each prompt the whole
growing transcript: throughput and TTFT on long prompts dominate, and prefix
caching (section 3) becomes essential. A reasoning model's hidden "thinking"
tokens make its effective TTFT a long decode.


# Prefill and decode

Generating text from a decoder-only transformer has two phases with opposite
performance characteristics. Understanding why one is compute-bound and the
other bandwidth-bound, using the roofline from lecture 12, explains every
serving technique that follows.


## One request, two phases

![A request is a single compute-heavy prefill over the whole prompt, then a loop of small decode steps, each producing one token. Source: instructor figure.](figs/15-Efficient-Inference/prefill-decode-timeline.png)

- **Prefill.** Run the whole prompt of $P$ tokens through the network in one
  forward pass, exactly as in training. Every matmul sees $P$ rows. Output:
  the first token, and the keys and values of every prompt token at every
  layer.
- **Decode.** Feed the last token in, get the next one out, repeat. Every
  matmul sees *one* row per sequence. Attention at step $t$ needs the keys and
  values of all $t-1$ earlier tokens, which is why they were kept.


## Arithmetic intensity of a decode step

Lecture 12's accounting for a bf16 matmul with $B$ token rows: $2Bd^2$ FLOPs,
$2d^2$ bytes of weights read, so $I \approx B$ FLOP/byte when $B \ll d$.
Apply it to a whole decode step for a batch of $B$ sequences and a model with
$N$ parameters:

$$
\begin{aligned}
\text{FLOPs} &\approx 2\,N\,B, \qquad
\text{bytes} \approx 2\,N \;(\text{bf16 weights}) + \text{KV cache read}, \\
I &\approx \frac{2NB}{2N} = B \;\;\text{FLOP/byte}.
\end{aligned}
$$

- **Every decode step reads every weight once**, and uses each weight $B$
  times. With $B = 1$ the intensity is 1 FLOP/byte against a ridge point of
  about 300 on an H100: the tensor cores sit idle 99.7% of the time.
- Decode is memory-bound until the batch reaches a few hundred sequences.
  Below that, time per step is set by **bytes over bandwidth**, not by FLOPs,
  and generating for 8 users costs about the same as generating for 1.
- Prefill is the same calculation with $B$ replaced by the prompt length $P$:
  a 4,096-token prompt sits on the compute roof. **Prefill is compute-bound;
  decode is bandwidth-bound.**


## The decode roofline

![Decode steps for Llama 3 8B placed on the H100 roofline by batch size, with bf16 and int4 weights, and a 4,096-token prefill for comparison. Source: instructor figure from datasheet numbers.](figs/15-Efficient-Inference/decode-roofline.png)


## Worked calculation: one decode step for Llama 3 8B

Llama 3 8B has $N = 8.0$B parameters
([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), Table 3), so
bf16 weights are $2N = 16$ GB.

1. **Lower bound on step time, batch 1, H100 (3.35 TB/s):**
   $16\text{ GB} / 3.35\text{ TB/s} = 4.8$ ms. At most about **210 tokens/s**
   for one user, however good the kernels are. On an A100 (2.0 TB/s): 8 ms,
   125 tokens/s.
2. **FLOPs, for comparison:** $2NB = 1.6\times10^{10}$ per step, which the
   H100 does in $16\,\mu\text{s}$ at peak. The arithmetic is 300× cheaper than
   the memory traffic.
3. **Batch 64:** the weights are still read once, so about the same 4.8 ms
   plus the KV-cache traffic (next slides); throughput is 64× higher for
   almost the same per-token latency. This is the whole case for batching.


Quizzable: given a model size, a weight precision and a GPU's bandwidth,
compute the decode-step time and the single-user tokens/s bound.


The same calculation in code. Change the model size, the bytes per parameter
or the GPU and rerun. The second table places a decode step on the roofline
for a range of batch sizes: the step time stays at the weight-read time until
$B$ passes the ridge point, and only then does the arithmetic start to cost
anything. KV-cache reads are left out, so these are lower bounds.

In [ ]:
N = 8.0e9               # parameters (Llama 3 8B)
bytes_per_param = 2     # bf16; 1 for int8/fp8, about 0.52 for int4 with g = 128
gpus = {                # dense bf16 peak (FLOP/s), HBM bandwidth (bytes/s)
    "H100 SXM": (989e12, 3.35e12),
    "A100 80GB": (312e12, 2.0e12),
}

weight_bytes = N * bytes_per_param
print(f"weights: {weight_bytes / 1e9:.0f} GB")
for name, (peak, bw) in gpus.items():
    t_mem = weight_bytes / bw          # every decode step reads every weight once
    t_flop = 2 * N / peak              # 2N FLOPs per token, at peak
    print(f"{name}: step >= {t_mem * 1e3:.1f} ms, at most {1 / t_mem:.0f} tokens/s "
          f"for one user; the FLOPs take {t_flop * 1e6:.0f} us ({t_mem / t_flop:.0f}x less)")

peak, bw = gpus["H100 SXM"]
ridge = peak / bw
print(f"\nH100 ridge point: {ridge:.0f} FLOP/byte")
print(f"{'batch B':>8} {'I (FLOP/byte)':>14} {'step (ms)':>10} {'tokens/s, all users':>20}  bound by")
for B in [1, 8, 64, 256, 512, 1024]:
    flops = 2 * N * B
    intensity = flops / weight_bytes   # = B for bf16 weights
    t_step = max(weight_bytes / bw, flops / peak)
    regime = "memory" if intensity < ridge else "compute"
    print(f"{B:>8} {intensity:>14.0f} {t_step * 1e3:>10.2f} {B / t_step:>20,.0f}  {regime}")

## The KV cache: what it stores

At layer $\ell$, head $h$, token $t$, attention needs $\mathbf{k}_{t}^{(\ell,h)}$
and $\mathbf{v}_{t}^{(\ell,h)}$, computed from the token's residual stream
(lecture 10). They do not change once computed, because the model is causal:
token $t$ never sees later tokens.

- Without a cache, step $t$ would recompute keys and values for all $t$ tokens
  at every layer: a prefill of length $t$ each step, $O(t^2)$ total.
- With a cache, step $t$ computes $\mathbf{k}_t, \mathbf{v}_t$ for the one new
  token, appends them, and attends over the stored $t-1$ others. Cost per step
  is $O(t)$ for attention and $O(N)$ for everything else.
- What is cached is **activations, not weights**: a tensor per sequence that
  grows by one position per generated token, lives for the request, and is
  thrown away afterwards. Its size is the second term in every serving memory
  budget, and at long context it is the first.


## KV-cache size per token, and why GQA and MLA matter for serving

Recall from lecture 11: for $L$ layers, $n_{kv}$ key/value heads of dimension
$d_\text{head}$, at $b$ bytes per value,
$\text{bytes per token} = 2 \cdot L \cdot n_{kv} \cdot d_\text{head} \cdot b$.

- **Llama 3 8B**: $L = 32$, $n_{kv} = 8$, $d_\text{head} = 128$, bf16:
  $2 \times 32 \times 8 \times 128 \times 2 = 131{,}072$ B $=$ **128 KiB per
  token**. At 8K context: $128\text{ KiB} \times 8{,}192 = 1$ GiB per
  sequence, so $(80 - 16)$ GB holds about 60 such sequences and nothing else;
  at 128K, 16 GiB per sequence. Context length is a memory cost that scales
  with *users × tokens*, not with the model.
- **GQA** (lecture 11) is why Llama 3 has $n_{kv} = 8$ rather than 32: the
  cache shrinks by $n_q / n_{kv}$, 4× for Llama 3, and a decode step reads
  that much less.
- **Multi-head latent attention** (DeepSeek-V2) caches a compressed latent
  $\mathbf{c}_t \in \mathbb{R}^{512}$ plus a 64-dimensional RoPE key per layer
  instead of the heads themselves, and reconstructs $\mathbf{K}, \mathbf{V}$
  with an up-projection that folds into the query and output weights
  ([DeepSeek-AI 2024](https://arxiv.org/abs/2405.04434), section 2.1). Per
  token per layer: $576$ values instead of $2 \times n_{kv} \times 128$.

![KV cache for one sequence against context length for three attention variants at the same depth. The attention variant sets the slope; nothing else does. Source: instructor figure, computed from the formula above.](figs/15-Efficient-Inference/kv-cache-growth.png)


The KV-cache budget in code. Memory sizes follow the text: GPU memory and
weights in decimal GB (as on datasheets), cache sizes in binary KiB and GiB
(as they fall out of powers of two). Try $n_{kv} = 32$ (no GQA), $b = 1$ (an
fp8 cache) or a 128K context.

In [ ]:
L, n_kv, d_head = 32, 8, 128   # Llama 3 8B: layers, KV heads, head dimension
b = 2                          # bytes per cached value: 2 for bf16, 1 for fp8
context = 8192                 # tokens per sequence
hbm = 80e9                     # bytes on the GPU (80 GB)
weights = 16e9                 # bytes of bf16 weights (16 GB)

KiB, GiB = 2**10, 2**30
kv_per_token = 2 * L * n_kv * d_head * b       # K and V, every layer, every KV head
kv_per_seq = kv_per_token * context
fit = (hbm - weights) / kv_per_seq

print(f"KV cache per token: {kv_per_token:,} B = {kv_per_token / KiB:.0f} KiB")
print(f"per {context:,}-token sequence: {kv_per_seq / GiB:.2f} GiB; "
      f"at 128K tokens: {kv_per_token * 131072 / GiB:.0f} GiB")
print(f"sequences that fit in {(hbm - weights) / 1e9:.0f} GB: {fit:.1f} "
      f"({int(fit)} whole sequences, no headroom)")

mha = 2 * L * 32 * d_head * b                  # Llama 2 7B shape: 32 KV heads
mla = L * (512 + 64) * b                       # DeepSeek-V2 latent + RoPE key, same depth
print(f"\nper token at the same depth: MHA {mha / KiB:.0f} KiB, "
      f"GQA {kv_per_token / KiB:.0f} KiB, MLA {mla / KiB:.0f} KiB")

## What fits on one GPU: weights plus cache

In [ ]:
#| echo: false
#| fig-cap: "Llama 3 8B on an 80 GB GPU, 8K tokens per sequence, under three precision choices. The number of concurrent sequences is what is left after the weights, divided by the per-sequence cache. Computed from the formulas in this section; no headroom for activations or fragmentation."
#| fig-alt: "Three horizontal stacked bars to an 80 GB line. bf16 weights and bf16 KV cache: 16 GB of weights then 59 concurrent sequences. int4 weights and bf16 KV cache: 4.1 GB then 70 sequences. int4 weights and fp8 KV cache: 4.1 GB then 141 sequences."
import matplotlib.pyplot as plt

N, hbm = 8.0e9, 80e9                     # Llama 3 8B parameters; GPU memory in bytes
kv_bf16 = 2 * 32 * 8 * 128 * 2 * 8192    # bytes per 8K-token sequence: 1 GiB
cases = [                                # label, weight bytes, KV bytes per sequence
    ("bf16 weights\nbf16 KV", N * 16 / 8, kv_bf16),
    ("int4 weights\nbf16 KV", N * 4.125 / 8, kv_bf16),
    ("int4 weights\nfp8 KV", N * 4.125 / 8, kv_bf16 / 2),
]

fig, ax = plt.subplots(figsize=(7.5, 3.4))
for i, (label, w, kv) in enumerate(cases):
    n = int((hbm - w) // kv)
    ax.barh(i, w / 1e9, color="#2a78d6", label="weights" if i == 0 else None)
    ax.barh(i, n * kv / 1e9, left=w / 1e9, color="#eb6834",
            label="KV cache, 8K tokens per sequence" if i == 0 else None)
    ax.text((w + n * kv) / 1e9 + 1, i, f"{n} concurrent\nsequences", va="center", fontsize=9.5)
ax.axvline(80, color="#8a8984", ls="--", lw=1)
ax.set_yticks(range(len(cases)), [c[0] for c in cases], fontsize=9.5)
ax.set_xlim(0, 100)
ax.set_xlabel("GB on one 80 GB GPU")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False, fontsize=9)
ax.invert_yaxis()
plt.show()

- The batch size a GPU can run, and therefore its throughput, is a memory
  question: $B_\text{max} = (\text{HBM} - \text{weights}) / \text{KV per sequence}$.
- Quantizing the **weights** (section 4) frees a fixed amount; quantizing the
  **cache** scales the number of users. At long context the cache wins.
- Real systems reserve 5 to 10% for activations and the CUDA context; vLLM's
  `gpu_memory_utilization` is this knob.


# Batching and serving systems

The previous section showed that a decode step costs nearly the same for one
sequence as for sixty-four, and that the cache budget sets how many sequences
fit. A serving system's job is to keep the batch full and the cache unfragmented
while requests arrive and finish at random. Three ideas from 2022–2023 define
how every production server does this today.


## Static batching and its waste

![Static batching runs a fixed group of requests until the longest finishes; continuous batching replaces each finished request at the next decode step. Source: instructor figure (schematic).](figs/15-Efficient-Inference/continuous-batching.png)

- **Static batching** groups $B$ requests, pads them to a common length, and
  runs until the longest finishes. The hatched cells are GPU time spent on
  nothing; with output lengths varying from a few tokens to thousands, most
  of the batch is idle most of the time.
- A new request has to wait for the whole batch to finish: TTFT is set by
  someone else's answer length.
- Request-level scheduling made sense for fixed-size inputs (images); it is
  the wrong granularity for generation.


## Continuous batching: Orca

[Yu et al. 2022](https://www.usenix.org/conference/osdi22/presentation/yu)
(Orca, OSDI) made two changes:

1. **Iteration-level scheduling.** The scheduler runs *one decode step* at a
   time. After each step, finished sequences leave the batch and waiting
   requests join it. A slot is never idle for longer than one step.
2. **Selective batching.** The matmuls (which do not care which sequence a
   row belongs to) run on all sequences' rows concatenated into one tall
   matrix; attention (which does care, because every sequence has its own
   cache and length) runs per sequence. No padding anywhere.


Reported: 36.9× the throughput of NVIDIA's FasterTransformer at the same
latency, on GPT-3 175B. Every serving engine since (vLLM, TensorRT-LLM,
SGLang, llama.cpp's server) schedules this way; "continuous batching" is the
name that stuck.


Two refinements of the same schedule, because prefill (compute-bound) and
decode (bandwidth-bound) interfere when they share a GPU:

- **Chunked prefill** splits a long prompt into pieces scheduled alongside
  decode steps, so a new user's prompt does not spike everyone's TPOT
  ([Agrawal et al. 2023](https://arxiv.org/abs/2308.16369)).
- **Disaggregation** puts prefill and decode on separate GPU pools and ships
  the KV cache between them: DistServe reports 7.4× more requests under the
  same latency targets ([Zhong et al. 2024](https://arxiv.org/abs/2401.09670)),
  and DeepSeek-V3 serves with a 32-GPU prefill unit and a 320-GPU decode unit
  ([DeepSeek-AI 2024](https://arxiv.org/abs/2412.19437), section 3.4).


## PagedAttention: the cache as virtual memory

![Keys and values live in fixed-size blocks that need not be contiguous; the attention kernel gathers them through a block table. Figure 5 of Kwon et al., 2023.](figs/15-Efficient-Inference/pagedattention-kwon-fig5.png)

- Before vLLM, a request's cache was one contiguous buffer sized for the
  *maximum* possible length. [Kwon et al. 2023](https://arxiv.org/abs/2309.06180)
  measured that only 20 to 38% of KV memory in existing systems held actual
  tokens; the rest was reserved-but-unused space and fragmentation.
- **PagedAttention** borrows paging from operating systems: the cache is cut
  into blocks of, say, 16 tokens; a request's logical blocks map to physical
  blocks anywhere in GPU memory via a per-request block table; a new physical
  block is allocated only when the last one fills.
- Waste falls to under one block per request, so more sequences fit, so the
  batch is bigger: 2 to 4× the throughput of FasterTransformer and Orca at the
  same latency.
- Because a physical block can appear in several block tables, sequences that
  share a prefix **share its blocks** with a reference count: parallel
  sampling, beam search and a system prompt shared by every user store the
  shared part once; a write into a shared block copies it first
  (copy-on-write), as an OS forks a process.
- The same block table lets the scheduler **swap** a preempted request's blocks
  to CPU memory or recompute them later, instead of killing it when memory
  runs out.


## Prefix caching

Sharing blocks across *simultaneous* requests generalizes to sharing across
*time*: keep the blocks of a finished request, keyed by a hash of their
tokens, and reuse them when a new request starts with the same tokens.

- The whole prefill of the shared prefix is skipped: TTFT drops from
  proportional-to-prompt to proportional-to-the-new-suffix.
- vLLM does this with per-block hashes ("automatic prefix caching"); SGLang
  keeps the prefixes in a radix tree so the longest shared prefix is found in
  one lookup ([Zheng et al. 2023](https://arxiv.org/abs/2312.07104)).
- This is the mechanism behind API-level **prompt caching**, which matches
  *exact prefixes* of tokens; lecture 21 works the agent cost example.


# Quantization

Batching fills the GPU with useful work; quantization shrinks the work itself.
Since decode is bound by bytes, storing weights (and the cache) in fewer bits
is the most direct speedup there is, and the one students will use first.


## What quantization is

Map a tensor of 16-bit floats to $b$-bit integers with a shared scale $s$ (and
optionally a zero point $z$):

$$
q = \mathrm{clip}\!\Big(\mathrm{round}\big(\tfrac{w}{s}\big) + z,\; 0,\; 2^b - 1\Big), \qquad
\hat w = s\,(q - z), \qquad
s = \frac{\max w - \min w}{2^b - 1}.
$$

- The error $|w - \hat w| \le s/2$ is set by the **range** of the group that
  shares a scale. One scale per tensor is cheap but one outlier stretches the
  range for everyone; one scale per **group** of $g = 32$ to $128$ weights
  along a row is the standard compromise.
- Group scales cost bits: 4-bit weights with a 16-bit scale per 128 weights
  are $4 + 16/128 = 4.125$ bits per weight; the formats on a later slide
  differ mostly in $g$ and in the scale's own format.
- **Post-training quantization (PTQ)** takes a trained checkpoint and a few
  hundred calibration samples; it is what everything in this section does.
  **Quantization-aware training** fine-tunes with the rounding in the loop and
  reaches lower bits, at the cost of a training run.


## Where the savings come from: the quizzable calculation

Weight-only quantization of Llama 3 8B from bf16 to 4-bit with $g = 128$:

1. **Memory.** $8.0\text{B} \times 4.125 / 8 = 4.1$ GB instead of 16 GB: fits a
   laptop GPU, or leaves 76 GB of an H100 for cache.
2. **Decode step bound.** $4.1\text{ GB} / 3.35\text{ TB/s} = 1.2$ ms instead of
   4.8 ms: up to about 4× the single-user tokens/s, from bandwidth alone.
3. **FLOPs: unchanged.** The kernel dequantizes each weight to bf16 on the fly
   and multiplies at bf16 rate. Weight-only quantization saves bytes, not
   arithmetic, so it helps where decode is memory-bound (small batch) and
   does nothing for prefill or large batches.
4. **Measured speedups are below the bound** because dequantization is not
   free and other kernels do not shrink: GPTQ reports 3.25× on an A100 and
   4.5× on an A6000 for 3-bit OPT-175B.


Quizzable: for any model, bit width and group size, compute the bytes, the
step-time bound, and state which regime (prefill, small-batch decode,
large-batch decode) benefits.


The calculation for every format in the table two slides on. Bits per weight
include the scales: $b + (\text{scale bits})/g$. The step time is the
bandwidth bound at batch 1; the FLOPs column would be the same in every row.

In [ ]:
N = 8.0e9          # parameters (Llama 3 8B)
bw = 3.35e12       # HBM bandwidth, bytes/s (H100 SXM)
hbm = 80e9         # GPU memory, bytes

formats = {        # name: (bits per weight, bits per scale, weights per scale)
    "bf16":                    (16, 0, 1),
    "int8, per channel":       (8, 16, 4096),   # one scale per 4096-wide row
    "int4, g = 128":           (4, 16, 128),
    "MXFP4, g = 32":           (4, 8, 32),
    "NVFP4, g = 16":           (4, 8, 16),      # the per-tensor fp32 scale is negligible
}

base = N * 2 / bw
print(f"{'format':<20} {'bits/weight':>11} {'weights (GB)':>12} {'step (ms)':>10} "
      f"{'tokens/s':>9} {'speedup':>8} {'left for cache (GB)':>20}")
for name, (bits, scale_bits, g) in formats.items():
    bits_per_weight = bits + scale_bits / g
    w = N * bits_per_weight / 8
    t = w / bw
    print(f"{name:<20} {bits_per_weight:>11.3f} {w / 1e9:>12.2f} {t * 1e3:>10.2f} "
          f"{1 / t:>9.0f} {base / t:>7.2f}x {(hbm - w) / 1e9:>20.1f}")

## Weight-only: outliers, GPTQ, AWQ

- **Round-to-nearest works to 8 bits, then breaks.** The reason is
  **outlier features**: a few hidden dimensions with values 20× larger than
  the rest, which appear in every transformer past about 6.7B parameters.
  LLM.int8() keeps those few columns in fp16 and quantizes the rest
  per-vector, with no loss up to 175B
  ([Dettmers et al. 2022](https://arxiv.org/abs/2208.07339)).
- **GPTQ**: quantize each layer to minimize the *output* error
  $\|\mathbf{W}\mathbf{X} - \hat{\mathbf{W}}\mathbf{X}\|^2$ on calibration data,
  not the weight error. Columns are rounded one at a time and the remaining
  unrounded weights are updated to compensate, using the Hessian
  $\mathbf{H} = 2\mathbf{X}\mathbf{X}^\top$ (the Optimal Brain Surgeon idea).
  175B parameters to 3 or 4 bits in about four GPU-hours
  ([Frantar et al. 2022](https://arxiv.org/abs/2210.17323)).
- **AWQ**: the weights that matter are the ones multiplying large
  activations. Find the 1% of channels with the largest activation magnitude
  and *scale them up* by $s > 1$ before rounding (and the matching activations
  down, which is exact), so their relative rounding error shrinks. No
  back-propagation, no reconstruction, and the search for $s$ is a
  one-dimensional grid ([Lin et al. 2023](https://arxiv.org/abs/2306.00978)).
- Both ship as 4-bit checkpoints on the Hub (`-GPTQ`, `-AWQ`), with kernels in
  vLLM and TensorRT-LLM. AWQ's TinyChat reports over 3× the speed of fp16 on
  desktop and mobile GPUs.


## Activations too: W8A8 and the fp8 path

- To use the GPU's **low-precision tensor cores** (2× the bf16 rate), both
  operands must be low precision: weights *and* activations. That is what
  speeds up prefill and large-batch decode, which weight-only cannot.
- Activations are harder than weights: the outlier channels are in the
  activations. **SmoothQuant** divides each activation channel by a factor
  $s_j$ and multiplies the matching weight row by $s_j$ (mathematically
  identical), "migrating" the difficulty into the weights, which tolerate it;
  int8 weights and activations with negligible loss, 1.56× speed, half the
  memory ([Xiao et al. 2022](https://arxiv.org/abs/2211.10438)).
- **fp8** (E4M3) is the simpler route on Hopper and later: a floating-point
  format has per-value exponent, so outliers hurt less, and per-tile scaling
  handles the rest (lecture 12 covered the training side). fp8 weights, fp8
  activations and fp8 cache is a common production setting for
  H100-class serving.
- What stays high precision is the same list as in training: normalization,
  softmax, the embedding and the output head.


## Formats and the hardware they need

| Format | Bits / weight (incl. scales) | Scale granularity | Tensor cores needed | Where it is used |
|---|---|---|---|---|
| int8 (per-channel) | 8 | per output channel | int8 (A100 and later) | LLM.int8(), SmoothQuant, W8A8 |
| int4, $g = 128$ | 4.125 | 128 weights, fp16 scale | none: dequantize to bf16 | GPTQ, AWQ, GGUF `Q4_K_M` |
| fp8 E4M3 | 8 | per tile or per tensor | fp8 (H100 and later) | weights, activations, KV cache |
| MXFP4 | 4.25 | 32 weights, 8-bit power-of-two scale | fp4 (Blackwell), else dequantize | gpt-oss MoE weights |
| NVFP4 | 4.5 | 16 weights, fp8 E4M3 scale, plus a per-tensor fp32 scale | fp4 (Blackwell) | Blackwell inference and training |

Sources: [Micikevicius et al. 2022](https://arxiv.org/abs/2209.05433) (fp8),
[OCP Microscaling spec 2023](https://www.opencompute.org/documents/ocp-microscaling-formats-mx-v1-0-spec-final-pdf) (MX),
[NVIDIA 2025](https://arxiv.org/abs/2509.25149) (NVFP4),
[OpenAI 2025](https://arxiv.org/abs/2508.10925) (gpt-oss).

- **Weight-only formats run anywhere** because the kernel dequantizes; what
  they need is a fast kernel, not special hardware. A laptop GPU runs a Q4
  GGUF.
- **Activation formats need matching tensor cores**: fp8 needs Hopper, fp4
  needs Blackwell. The NVFP4 paper reports a 12B model pretrained on 10T
  tokens in 4-bit with accuracy matching fp8.
- gpt-oss-120b ships its MoE weights in MXFP4 so the 117B-parameter model fits
  one 80 GB GPU (60.8 GiB checkpoint).


## Quantizing the KV cache

The cache is activations, so it shares the activation problem (outlier
channels in the keys) and the activation benefit (it scales with users ×
context).

- **fp8 cache** halves it with no measurable loss and is a one-flag change in
  vLLM (`--kv-cache-dtype fp8`).
- **Keys deserve more bits than values.** Under a fixed memory budget, 4-bit
  keys with 2-bit values retain up to 98.3% of the accuracy of 4-bit for both,
  while reversing the allocation loses badly
  ([Hariri et al. 2025](https://arxiv.org/abs/2502.15075)). Sub-4-bit methods
  (KIVI, mixed precision for sensitive key channels) are in Further reading.
- As of late 2026 no major serving engine defaults to a sub-8-bit KV cache
  (vLLM: `kv_cache_dtype="auto"`); fp8 is one flag away.


## The accuracy picture

![Schematic of how quality degrades with bits per weight for naive and calibrated quantization. The shapes are a sketch of what the GPTQ, AWQ and llama.cpp evaluations show; the numbers are not measurements. Source: instructor figure.](figs/15-Efficient-Inference/quantization-accuracy-schematic.png)

- 8 bits is free. 4 bits with a calibrated method and $g \le 128$ costs a few
  tenths of a point of perplexity on a 7B model and less on larger models,
  which have more redundancy. 3 bits is where methods separate; 2 bits needs
  QAT or mixed precision.
- The loss is not uniform across tasks: long-context retrieval, math and code
  degrade before chat does. Measure on *your* task, not on perplexity alone.
- Smaller models lose more: a 4-bit 70B beats a bf16 8B, but a 4-bit 1B may
  not beat an 8-bit 0.5B. For a fixed memory budget, more parameters at fewer
  bits usually wins down to 4 bits
  ([Dettmers and Zettlemoyer 2023](https://arxiv.org/abs/2212.09720)).


## QLoRA: the bridge to fine-tuning on one GPU

Lecture 12's accounting says fine-tuning a 65B model needs about 1 TB of
optimizer state. [Dettmers et al. 2023](https://arxiv.org/abs/2305.14314)
fine-tuned one on a single 48 GB GPU:

1. **Freeze the base model in 4-bit NormalFloat (NF4)**, a 4-bit format whose
   16 levels are the quantiles of a normal distribution, which is what trained
   weights look like. Only the frozen weights are 4-bit; the matmul dequantizes
   to bf16.
2. **Train LoRA adapters** ($\mathbf{W} + \mathbf{B}\mathbf{A}$, rank 8 to 64,
   [Hu et al. 2021](https://arxiv.org/abs/2106.09685)) in bf16 on every linear
   layer; gradients flow *through* the frozen 4-bit weights into the adapters.
   Optimizer state exists only for the adapters: megabytes, not gigabytes.
3. **Double quantization** of the scales and **paged optimizer** states that
   spill to CPU memory on spikes.


Guanaco, trained this way for 24 hours on one GPU, reached 99.3% of ChatGPT's
score on the Vicuna benchmark. QLoRA is how a student fine-tunes an 8B model on
one SCC GPU for the final project, and the memory arithmetic is this
lecture's: $0.5$ bytes per base parameter plus activations.


# Speculative decoding

Quantization reduces bytes per step. Speculative decoding attacks the other
factor: the number of steps. Because a decode step for one token costs the same
bandwidth as a step for several, verifying several guessed tokens in one pass
is almost free, as long as the guesses are usually right.


## Draft, then verify

![Each row is one iteration: a small draft model proposes tokens (green when accepted), the target model rejects the first wrong one (red) and supplies its own token (blue) in the same forward pass. Figure 1 of Leviathan et al., 2023.](figs/15-Efficient-Inference/speculative-decoding-leviathan-fig1.png)

- A small **draft** model $M_q$ generates $\gamma$ tokens autoregressively
  (cheap: it is small). The **target** model $M_p$ then runs *one* forward
  pass on all $\gamma$ candidates at once, exactly like a prefill, producing
  its own next-token distribution at every position.
- Accept the draft's tokens left to right until the first one the target
  "disagrees" with; at that position, sample from the target instead. One
  target pass yields between 1 and $\gamma + 1$ tokens.
- The target pass over $\gamma + 1$ tokens costs almost the same *time* as
  a pass over 1 (section 2: memory-bound), so each accepted token is nearly
  free. [Leviathan et al. 2023](https://arxiv.org/abs/2211.17192) report
  2 to 3× on T5-XXL with identical outputs;
  [Chen et al. 2023](https://arxiv.org/abs/2302.01318) 2 to 2.5× on
  Chinchilla 70B.


## The acceptance rule, and why the output distribution is exact

At one position the draft proposes $x \sim q(x)$ and the target has $p(x)$.
**Accept** $x$ with probability $\min\big(1, p(x)/q(x)\big)$; on rejection,
**resample** from the residual $p'(x) = \max\big(0,\, p(x) - q(x)\big) / Z$.
Claim: the token produced is distributed exactly as $p$.

$$
\begin{aligned}
\Pr(\text{output} = x) &= \underbrace{q(x)\min\!\Big(1, \tfrac{p(x)}{q(x)}\Big)}_{\text{proposed and accepted}}
 + \underbrace{\Big(1 - \textstyle\sum_{x'} \min(p(x'), q(x'))\Big)}_{\Pr(\text{reject})} \cdot \frac{\max(0, p(x) - q(x))}{Z} \\[2pt]
&= \min(p(x), q(x)) + \max(0,\, p(x) - q(x)) \;=\; p(x),
\end{aligned}
$$
because $Z = \sum_{x'} \max(0, p(x') - q(x')) = \sum_{x'} \big(p(x') - \min(p(x'), q(x'))\big) = 1 - \sum_{x'} \min(p, q)$, which cancels the rejection probability.


Consequences: speculative decoding is **lossless** for any draft model,
including a bad one; the draft only affects *speed*. The acceptance
probability of a position is $\beta = \sum_x \min(p(x), q(x))$, one minus a
total-variation-like divergence between $p$ and $q$
([Leviathan et al. 2023](https://arxiv.org/abs/2211.17192), Theorem 3.5). The
same rule appears independently in Chen et al. as "speculative sampling".
Quizzable: reproduce this two-line proof.


A numerical check of the proof. Draw many proposals from a deliberately poor
draft $q$, apply the accept-or-resample rule, and compare the histogram of
the output tokens with the target $p$. The output matches $p$, not $q$, up to
sampling noise, and the fraction accepted matches $\beta = \sum_x \min(p, q)$.
Change `q` to anything with full support and the output still matches `p`;
only the acceptance rate moves.

In [ ]:
#| code-fold: true
#| fig-cap: "Speculative sampling at one position over a six-token vocabulary: the draft distribution $q$, the target $p$, and the empirical distribution of 200,000 tokens produced by the accept-or-resample rule."
#| fig-alt: "Grouped bar chart over tokens 0 to 5. The draft q is flat-ish with most mass on tokens 1 and 2; the target p decreases from 0.40 at token 0 to 0.04 at token 5. The empirical output bars sit on top of the p bars at every token."
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
p = np.array([0.40, 0.25, 0.15, 0.10, 0.06, 0.04])   # target model's next-token distribution
q = np.array([0.10, 0.30, 0.30, 0.05, 0.15, 0.10])   # draft model's (a poor draft)
n = 200_000

x = rng.choice(len(q), size=n, p=q)                   # draft proposals
accept = rng.random(n) < np.minimum(1.0, p[x] / q[x])
residual = np.maximum(0.0, p - q)
residual /= residual.sum()                            # p' = max(0, p - q) / Z
resampled = rng.choice(len(p), size=n, p=residual)
out = np.where(accept, x, resampled)

empirical = np.bincount(out, minlength=len(p)) / n
beta = np.minimum(p, q).sum()
print(f"acceptance rate: measured {accept.mean():.4f}, beta = sum min(p, q) = {beta:.4f}")
print(f"max |empirical - p| = {np.abs(empirical - p).max():.4f}  "
      f"(sampling noise is about {np.sqrt(p.max() * (1 - p.max()) / n):.4f})")
print(f"max |empirical - q| = {np.abs(empirical - q).max():.4f}")

tokens = np.arange(len(p))
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(tokens - 0.27, q, width=0.27, color="#8a8984", label="draft $q$")
ax.bar(tokens, p, width=0.27, color="#2a78d6", label="target $p$")
ax.bar(tokens + 0.27, empirical, width=0.27, color="#eb6834", label="output of the rule")
ax.set_xticks(tokens)
ax.set_xlabel("token")
ax.set_ylabel("probability")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

## Expected speedup as a function of acceptance rate

With acceptance rate $\alpha = \mathbb{E}[\beta]$ treated as i.i.d. across
positions, the number of tokens from one target pass is a geometric variable
capped at $\gamma + 1$:

$$
\mathbb{E}[\text{tokens per target pass}] = \frac{1 - \alpha^{\gamma+1}}{1 - \alpha},
\qquad
\text{speedup} = \frac{1 - \alpha^{\gamma+1}}{(1 - \alpha)(\gamma c + 1)},
$$

where $c$ is a draft step's cost as a fraction of a target step
([Leviathan et al. 2023](https://arxiv.org/abs/2211.17192), eq. 1 and Theorem 3.8).

In [ ]:
#| echo: false
#| fig-cap: "Left: expected tokens per target pass. Right: wall-clock speedup at $\\gamma = 4$ for three draft costs. Computed from the formulas above."
#| fig-alt: "Two panels. Left: expected tokens per target forward pass rising with acceptance rate alpha for gamma 2, 4 and 8, reaching about 3, 5 and 9 as alpha approaches 1. Right: speedup against alpha at gamma 4 for draft costs 0.01, 0.05 and 0.2; the cheap draft reaches about 4.7 at alpha near 1, the expensive one crosses 1 only above alpha of about 0.45."
import numpy as np
import matplotlib.pyplot as plt

def expected_tokens(alpha, gamma):
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)       # Leviathan et al., eq. 1

def speedup(alpha, gamma, c):
    return expected_tokens(alpha, gamma) / (gamma * c + 1)   # Theorem 3.8

alpha = np.linspace(0, 0.99, 300)
colors = ["#2a78d6", "#eb6834", "#1baf7a"]
fig, axes = plt.subplots(1, 2, figsize=(9, 2.9))
for g, color in zip([2, 4, 8], colors):
    axes[0].plot(alpha, expected_tokens(alpha, g), lw=2, color=color, label=f"$\\gamma = {g}$")
axes[0].set_ylabel("expected tokens per target pass")
for c, color in zip([0.01, 0.05, 0.2], colors):
    axes[1].plot(alpha, speedup(alpha, 4, c), lw=2, color=color, label=f"$c = {c}$")
axes[1].axhline(1, color="#8a8984", ls="--", lw=1)
axes[1].set_ylabel(r"wall-clock speedup at $\gamma = 4$")
for ax in axes:
    ax.set_xlabel(r"acceptance rate $\alpha$")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
plt.show()

The quizzable numbers, and the break-even acceptance rate for a given draft
cost (where the speedup equals 1), found by bisection. Change $\gamma$ to see
that a longer draft needs a higher $\alpha$ to pay for itself.

In [ ]:
alpha, gamma, c = 0.8, 4, 0.05   # acceptance rate, draft length, draft cost / target cost

def expected_tokens(alpha, gamma):
    return (1 - alpha ** (gamma + 1)) / (1 - alpha)

def speedup(alpha, gamma, c):
    return expected_tokens(alpha, gamma) / (gamma * c + 1)

def break_even_alpha(gamma, c):
    lo, hi = 0.0, 0.999999       # speedup increases with alpha
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if speedup(mid, gamma, c) < 1 else (lo, mid)
    return lo

print(f"alpha = {alpha}, gamma = {gamma}, c = {c}: "
      f"{expected_tokens(alpha, gamma):.2f} tokens per target pass, "
      f"speedup {speedup(alpha, gamma, c):.2f}x")
for c_draft in [0.01, 0.05, 0.2]:
    print(f"c = {c_draft}: break-even alpha = {break_even_alpha(gamma, c_draft):.3f}, "
          f"speedup at alpha = 0.99: {speedup(0.99, gamma, c_draft):.2f}x")

## Variants, and when it does not help

- **Drafters without a second model.** Medusa bolts extra output heads onto
  the target ([Cai et al. 2024](https://arxiv.org/abs/2401.10774)); EAGLE
  drafts in the target's hidden-state space with one small head, 2.7 to 3.5×
  on Llama 2 Chat 70B ([Li et al. 2024](https://arxiv.org/abs/2401.15077)),
  and is what vLLM and SGLang ship.
- **Where it stops paying.** The gain comes from spare bandwidth-bound
  capacity. At large batch, decode is near the compute roof, the verification
  FLOPs for rejected tokens are real, and speculation can *slow* a server
  down. It is a latency tool for small batches and interactive use, not a
  throughput tool; measure $\alpha$ on your workload before turning it on.


# Smaller models

Everything so far serves a fixed model faster. The remaining lever is to serve
a smaller model that is good enough, which raises the question of how to make a
small model good: distillation. This section also covers the serving cost of
mixture-of-experts models, which are "small" per token but not in memory, and
the engineering decision that every LLM application faces.


## Knowledge distillation

[Hinton et al. 2015](https://arxiv.org/abs/1503.02531): train a small
**student** to match a large **teacher's** output distribution, not just the
labels. With logits $z_i$ and a temperature $T$,

$$
\mathrm{Pr}(y = i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}, \qquad
L = T^2\, \mathrm{KL}\big(\mathrm{Pr}_\text{teacher}^{(T)} \,\|\, \mathrm{Pr}_\text{student}^{(T)}\big) + \lambda\, L_\text{CE}(\text{labels}).
$$

- The soft targets carry **dark knowledge**: which wrong answers are nearly
  right. A high $T$ flattens the distribution so those small probabilities
  contribute to the gradient; $T^2$ keeps the gradient scale constant.
- For LLMs, the per-token version is cheap when teacher logits are available:
  Gemma 2's 2B and 9B models were *pretrained* this way on the logits of a
  larger model, which the authors found better than training on the same
  tokens with hard labels ([Gemma Team 2024](https://arxiv.org/abs/2408.00118)).
- Distillation is the honest version of "a 4-bit 70B beats an 8B": you cannot
  get the teacher's quality in the student, but you get much more than the
  student would learn alone with the same data.


## Sequence-level distillation and the R1-Distill models

- **Sequence-level distillation**: instead of matching the teacher's
  distribution at every token, generate whole outputs from the teacher and
  fine-tune the student on them with ordinary cross-entropy
  ([Kim and Rush 2016](https://arxiv.org/abs/1606.07947)). It needs only
  teacher *samples*, so it works through an API, and it is what everyone
  means by "distillation" in 2026.
- **DeepSeek-R1-Distill**: about 800k reasoning traces curated from
  DeepSeek-R1 were used to fine-tune Qwen 2.5 and Llama 3 bases from 1.5B to
  70B parameters, with supervised fine-tuning only and no RL stage
  ([DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), section 2.4).
- Their comparison: running R1's RL recipe directly on a 32B model gave a
  worse model than distilling R1 into the same 32B (section 4.1). The
  reasoning patterns that RL discovers at 671B transfer to a small model more
  cheaply than the small model could discover them.
- Serving consequence: an R1-Distill-7B answers a math question at a 7B
  model's cost, which is what makes reasoning affordable on one GPU, with the
  caveat that it still emits thousands of thinking tokens.


## Mixture of experts: cheap per token, not in memory

Lecture 11 owns MoE as architecture (DeepSeek-V3: 671B parameters, 37B active
per token). The two serving facts:

- **All experts must be resident.** Routing is decided at run time, so every
  expert's weights sit in GPU memory: 671B × 1 byte (fp8) $\approx$ 671 GB,
  at least nine 80 GB GPUs before any cache. Per-token FLOPs are a 37B
  model's; bytes are a 671B model's.
- **MoE pays off at large batch with expert parallelism.** A batch of 64
  touches most experts, so at small batch decode reads nearly the whole model
  per step; spreading experts over GPUs so each reads only its own is why
  DeepSeek-V3's decode unit is 320 GPUs
  ([DeepSeek-AI 2024](https://arxiv.org/abs/2412.19437), section 3.4). On one
  GPU, a dense 8B is usually the better choice.


## Fine-tune, prompt, or retrieve: a cost and latency question

| Option | Pays | Every call costs | Latency | Changes knowledge by |
|---|---|---|---|---|
| **Prompt** (instructions, few-shot examples) | nothing up front | the prompt's tokens, every call (prefix caching cuts most of it) | prefill of the prompt | editing text |
| **RAG** (lecture 13) | an index | retrieval + the retrieved tokens | retrieval + a longer prefill | re-indexing, minutes |
| **Fine-tune** (LoRA / QLoRA) | one training run | a short prompt on a possibly smaller model | the shortest | retraining, hours |

Lecture 13 has the framing of when each wins; this table is the cost side.

# Practicalities

The final project asks you to run, measure and report an open-weight model.
This section is the checklist. Tool and model names will date; the measurements
and the arithmetic will not.


## Running an open-weight model

| Tool | Best for | Formats | Notes |
|---|---|---|---|
| **vLLM** | a GPU server, many concurrent requests | bf16, fp8, GPTQ, AWQ | PagedAttention, continuous batching, prefix caching, speculative decoding; OpenAI-compatible API |
| **SGLang** | the same, with structured output and agent workloads | as vLLM | RadixAttention prefix cache |
| **llama.cpp** | a laptop, CPU or any single GPU | GGUF (`Q4_K_M`, `Q8_0`, ...) | runs anywhere; `llama-bench` for measurement |
| **Ollama** | the easiest first run | GGUF | a wrapper over llama.cpp with a model library |

```sh
# One SCC GPU: a dense 8B model from the current Qwen / Llama / Gemma generation
vllm serve Qwen/Qwen3-8B --max-model-len 16384 --gpu-memory-utilization 0.9
# A laptop: a 4-bit GGUF of the same model
ollama run qwen3:8b
```

Qwen3-8B is the example; substitute this year's small open-weight release.
The SCC's A100s have no fp8 tensor cores, so use bf16 or AWQ/GPTQ int4
there; its L40S cards (Ada) do support fp8.

## Choosing a quantization

1. **Does bf16 fit with room for the cache?** Weights $2N$ bytes plus
   KV per sequence × concurrent sequences plus 10%. If yes, start there; it
   is the reference you will compare against.
2. **If not, 8-bit first** (fp8 on Hopper+, int8 or `Q8_0` elsewhere). It is
   free in quality and halves the bytes.
3. **Then 4-bit, calibrated, $g \le 128$** (AWQ or GPTQ for vLLM, `Q4_K_M` for
   llama.cpp). Expect a few tenths of a perplexity point on an 8B model, more
   on smaller ones; check your task.
4. **Prefer more parameters at 4 bits to fewer at 16** for the same memory,
   down to about 4 bits. Below that, stop.
5. **Quantize the cache (fp8) before going below 4-bit weights** when context
   or concurrency is the constraint; it is the cheaper trade.


Always run the same evaluation on the quantized and unquantized model. A
quantization that is "lossless on perplexity" can still break tool-call
formatting or long-context recall.


## Measuring tokens per second

- **Measure three things separately**: TTFT at a fixed prompt length,
  single-stream decode tokens/s (TPOT), and aggregate throughput at a stated
  concurrency. A tokens/s number without the batch size and prompt length is
  meaningless.
- **Warm up**, then time at least a few hundred output tokens; the first
  request pays for kernel compilation and cache allocation.
- **Check against the bound.** Single-stream decode tokens/s cannot exceed
  bandwidth ÷ (weight bytes + cache bytes read per step). If you measure 60
  tokens/s on an A100 for a bf16 8B model, you are at about half the 125
  tokens/s bound, which is normal; if you measure 20, something is wrong
  (CPU offload, a slow kernel, a missing attention backend).
- Tools: `vllm bench throughput` / `vllm bench serve` for a server,
  `llama-bench` for llama.cpp, or thirty lines of Python against the
  OpenAI-compatible endpoint, recording the timestamps of streamed tokens.
- Report the GPU, the precision of weights *and* cache, the context length,
  the serving engine and version, and the concurrency. Those five settings
  move the number by 10×.


## Cost worked example: tokens per dollar

A GPU rented at $P$ dollars per hour, delivering an aggregate throughput of
$T$ tokens/s:

$$
\text{cost per million tokens} = \frac{P \times 10^6}{3600\,T} \approx \frac{278\,P}{T} \text{ dollars}.
$$

- An H100 at about \$2 per hour (the DeepSeek-V3 paper's assumption, §1, which
  lecture 12 used)
  serving Llama 3 8B:
- **Single stream**, 150 tokens/s: $278 \times 2 / 150 = $ **\$3.70 per
  million tokens**. The GPU is 99.7% idle.
- **Batch 64**, about 5,000 tokens/s aggregate (the weight read is shared and
  the cache read grows with the batch): **\$0.11 per million tokens**, 33×
  cheaper per token, from the same hardware.
- **int4 weights, fp8 cache, batch 128**: roughly double again. This is the
  arithmetic behind the price of small-model API tokens, and the reason
  providers batch aggressively and quote per-token prices far below what a
  dedicated GPU would cost you.


Quizzable: given $P$, a measured $T$ and a batch size, compute the cost per
million tokens, and say which of the lecture's techniques would move it most.


The cost formula in code, with the two scenarios above. Put in your own GPU
price and the throughput you measure for the final project.

In [ ]:
# A million tokens at T tokens/s take 1e6 / T seconds = 1e6 / (3600 T) hours.
# At P dollars per hour that costs P * 1e6 / (3600 T) = (1e6 / 3600) * P / T,
# and 1e6 / 3600 = 277.8, the "278" in the formula.
P = 2.00                         # dollars per GPU-hour (H100)
scenarios = {                    # name: aggregate tokens/s on one GPU
    "single stream": 150,
    "batch 64": 5000,
}

print(f"1e6 / 3600 = {1e6 / 3600:.1f}")
cost = {name: P * 1e6 / (3600 * T) for name, T in scenarios.items()}
for name, T in scenarios.items():
    print(f"{name:>14}: {T:>5,} tokens/s -> ${cost[name]:.2f} per million tokens")
print(f"batching is {cost['single stream'] / cost['batch 64']:.0f}x cheaper per token")

## What to report in the final project

For any model you serve, fine-tune or evaluate, one table:

| | |
|---|---|
| Model and size | name, parameters, dense or MoE, $n_{kv}$ and $L$ |
| Precision | weights, activations, KV cache; method and group size |
| Hardware | GPU model, HBM, bandwidth; one GPU or several and how split |
| Memory | weights + cache at your context length, from the formulas, and as measured |
| Latency | TTFT at your prompt length; TPOT single-stream |
| Throughput | tokens/s at a stated concurrency, and the bandwidth bound it is compared to |
| Cost | dollars per million tokens at a stated GPU price |
| Quality | the same evaluation before and after every precision or model change |

The point is not the numbers but the comparison: measured against the bound,
quantized against unquantized, small model against large. A project that
reports "4-bit was 3.1× faster at 0.2 perplexity cost, against a 4× bandwidth
bound" has understood this lecture.


# Summary

One line per idea; the recap deck uses these.


## Summary

- Training is one compute-bound job; serving is millions of small ones, costed per generated token, and judged by TTFT, TPOT, throughput and goodput.
- Prefill runs the whole prompt at once and is compute-bound; decode produces one token per step, reads every weight each step, and has intensity $I \approx B$ FLOP/byte, so it is memory-bandwidth-bound until the batch reaches a few hundred.
- Decode-step time is bounded below by bytes over bandwidth: 16 GB of bf16 Llama 3 8B weights over 3.35 TB/s is 4.8 ms, about 210 tokens/s for one user on an H100.
- The KV cache stores every layer's keys and values for every token: $2\,L\,n_{kv}\,d_\text{head}\,b$ bytes per token, 128 KiB for Llama 3 8B, 1 GiB per 8K sequence; GQA and MLA shrink it by sharing or compressing heads.
- Continuous batching (Orca) schedules per decode step and batches only the matmuls; PagedAttention (vLLM) stores the cache in blocks through a block table, so no memory is reserved for tokens that may never come, and prefixes are shared.
- Prefix caching reuses the blocks of an earlier identical prefix, which is why agent harnesses put stable content first; disaggregation runs prefill and decode on separate GPU pools.
- Quantization maps weights to $b$-bit integers with a per-group scale; weight-only 4-bit cuts bytes 4× and speeds up memory-bound decode, W8A8 and fp8 use low-precision tensor cores and speed up prefill too; outlier channels are why calibration (GPTQ, AWQ, SmoothQuant) is needed below 8 bits.
- The KV cache quantizes to fp8 for free and to 4 bits with care; keys need more bits than values.
- QLoRA freezes the base model in NF4 and trains LoRA adapters, so an 8B model fine-tunes on one GPU.
- Speculative decoding drafts $\gamma$ tokens with a cheap model and verifies them in one target pass; the accept-with-$\min(1, p/q)$-else-resample-from-$\max(0, p-q)$ rule makes the output exactly the target's distribution, and the expected tokens per pass are $(1-\alpha^{\gamma+1})/(1-\alpha)$.
- Distillation trains a small student on a teacher's soft targets or sampled outputs; the R1-Distill models show that reasoning transfers to small models more cheaply by distillation than by RL.
- MoE models are cheap per token but need every expert resident, so they pay off at large batch with expert parallelism, not on one GPU.
- Prompting pays per call, RAG pays for retrieval and is how to inject changing facts, fine-tuning pays once; prefix caching has shifted the break-even toward prompting.
- Cost per million tokens is $278\,P/T$; batching, then quantization, is what moves it.

## Further reading

- [Austin et al., *How to Scale Your Model* (Google DeepMind, 2025)](https://jax-ml.github.io/scaling-book/), the inference chapters: roofline analysis of prefill, decode and the KV cache, with worked numbers.
- [Kwon et al., *Efficient Memory Management for LLM Serving with PagedAttention* (2023)](https://arxiv.org/abs/2309.06180): the vLLM paper; sections 3 and 4 are the clearest account of why the cache is the problem.
- [Leviathan et al., *Fast Inference from Transformers via Speculative Decoding* (2023)](https://arxiv.org/abs/2211.17192): short, and the proofs are a page.
- [Dettmers et al., *QLoRA* (2023)](https://arxiv.org/abs/2305.14314) and [Lin et al., *AWQ* (2023)](https://arxiv.org/abs/2306.00978): the two quantization papers whose code you will actually run.
- [Zhong et al., *DistServe* (2024)](https://arxiv.org/abs/2401.09670): goodput, and why prefill and decode want different machines.
- KV-cache quantization below 4 bits: [Liu et al., *KIVI* (2024)](https://arxiv.org/abs/2402.02750) (2-bit, keys per channel and values per token, 2.6× less peak memory and 2.35 to 3.47× throughput) and [Xia et al. (2025)](https://arxiv.org/abs/2511.18643) (mixed precision that keeps a few sensitive key channels wider, about 8× smaller at negligible loss).
- The [vLLM documentation](https://docs.vllm.ai/) and the [llama.cpp repository](https://github.com/ggml-org/llama.cpp): the current flags, formats and benchmarks.